### SCPT to VS all-in-one tool (Beta)

In [1]:
# Import packages for use:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('display.max_columns', None)

# Install the package first: pip install cpt-vs-tool (or pip install -e . from the repository root)
from cpt_vs_tool import compute_profiles, interleave, compute_VsZ, a_coeff


### IMPORT TRAVEL TIME DATA

In [2]:
'''

read travel time data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of travel time is millisecond.
change the name to the target input file. 

'''

# TT_DATA = pd.read_csv('SCPT-82-tt.csv')
TT_DATA = pd.read_csv('UCD_3_SCPT_tt.csv')

TT_depth = np.asarray(TT_DATA['depth (m)'])
Meas_TT = np.asarray(TT_DATA['traveltime (ms)'])

### IMPORT CPT DATA

In [3]:
'''
read CPT data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of qt and fs are MPa.
change the name to the target input file. 

'''

# CPT_DATA = pd.read_csv('SCPT-82.csv')
CPT_DATA = pd.read_csv('UCD_3_SCPT.csv')

CPT_depth = np.asarray(CPT_DATA['depth (m)'])
CPT_qt = np.asarray(CPT_DATA['qt (MPa)'])
CPT_fs = np.asarray(CPT_DATA['fs (MPa)'])
CPT_qc = np.asarray(CPT_DATA['qc (MPa)'])

valid_mask = ~np.isnan(CPT_depth) & ~np.isnan(CPT_qt) & ~np.isnan(CPT_fs)

CPT_depth = CPT_depth[valid_mask]
CPT_qt = CPT_qt[valid_mask]
CPT_fs = CPT_fs[valid_mask]
CPT_qc = CPT_qc[valid_mask]

# Enter depth of groundwater table in meter, if the groundwater table is not encountered, please enter a value larger than the maximum depth of CPT data.
print (np.max (CPT_depth))
dGWT = 3

14.611


### IMPORT SL DATA

In [4]:
SL_DATA = pd.read_csv('UCD_3.csv')
# SL_DATA = pd.read_csv('RA-1.csv')

SL_depth = np.asarray(SL_DATA['depth (m)'])
SL_Vs = np.asarray(SL_DATA['Vs (m/s)'])

In [5]:
"""Enter the geology type of the site, the options are the keys of cpt_vs_tool.a_coeff:
'Beach and Dune', 'Floodplain', 'Glacial', 'Lagoonal', 'Valley Train', 'af/Qi', 'Qal1', 'Qal2', 'Qal3', 'Qoa', 'Qs'"""

geology = "Qoa"

p = compute_profiles(CPT_depth, CPT_qt, CPT_fs, dGWT, TT_depth, Meas_TT, geology)
print(f"geology = {geology}  |  a_geo = {p['a_geo']:.4f}  |  a_ss = {p['a_site_specific']:.4f}")

VsZ_results = compute_VsZ(p)
pd.DataFrame([{'Method': k, 'Z (m)': Z, 'VsZ (m/s)': v} for k, (v, Z) in VsZ_results.items()])

geology = Qoa  |  a_geo = 4.5154  |  a_ss = 4.5640


,Method,Z (m),VsZ (m/s)
0,Zhang et al.,14.611000,293.245238
1,Zhang et al. (ss),14.611000,307.831781
2,Robertson (2012),14.611000,223.610966
3,LBR,14.398752,310.592588


##### Export site specific results to CSV

In [6]:
# Export results to CSV
site_specific_results = pd.DataFrame({
    "depth (m)": p['CPT_depth'],
    "velocity (m/s)": p['site_specific_velocity']
})
site_specific_results.to_csv('vs_ss_results.csv', index=False)

In [9]:
def plot_profiles(p):
    CPT_depth, Qtn, Qtncs, sigmavp, Ic = p['CPT_depth'], p['Qtn'], p['Qtncs'], p['sigmavp'], p['Ic']
    TT, TT_depth, halfdepth, DTT = p['TT'], p['TT_depth'], p['halfdepth'], p['DTT']
    results = p['results']

    fig, axs = plt.subplots(1, 6, figsize=(12, 5), dpi = 300)

    axs[0].plot(Qtn, CPT_depth, label='Qtn', color = 'grey', linewidth = 1.2)
    axs[0].plot(Qtncs, CPT_depth, label='Qtncs', color='tab:blue', linewidth=1.2)
    axs[0].set_xlim(0, 1000)
    axs[0].set_xticks(np.arange(0, 1001, 200))
    axs[0].grid(True, which="both", ls="--")
    axs[0].set_xlabel('Qtn / Qtncs')
    axs[0].set_ylabel('Depth (m)')
    axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[1].plot((0, sigmavp[0]), (0, CPT_depth[0]), color='tab:blue', linestyle='dashed', linewidth = 1.2)
    axs[1].plot(sigmavp, CPT_depth, label='$\\sigma_v^\\prime$_CPT', color='tab:blue', linewidth = 1.2)
    axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
    axs[1].set_xticks(np.arange(0, 11, 2))
    axs[1].set_xlim(0, 10)
    axs[1].grid(True, which="both", ls="--")
    axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[2].plot(Ic, CPT_depth, label='$I_c$', color = 'tab:blue', linewidth = 1.2)
    axs[2].set_xlim(1, 4)
    axs[2].grid(True, which="both", ls="--")
    axs[2].set_xlabel('Ic')
    axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[3].scatter(DTT, halfdepth, label='Measured $\\Delta$tt', marker='o', edgecolor='black', color = 'white', s=10)
    axs[3].scatter(p['tt_hat_global'], p['halfdepth_global'], label='Predicted $\\Delta$tt', marker='x', color='tab:blue', s=10)
    axs[3].scatter(p['tt_hat_ss'], p['halfdepth_ss'], label='Predicted $\\Delta$tt (ss)', marker='x', color='tab:orange', s=10)
    axs[3].set_xlabel('$\\Delta$tt (ms)')
    axs[3].set_xlim(0, 10)
    axs[3].set_xticks(np.arange(0, 11, 2))
    axs[3].grid(True, which="both", ls="--")
    axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[4].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)
    colors = plt.colormaps['tab10']

    # Plot each segment with a different color
    for i in range(len(results['breaks']) - 1):
        # Define segment depth range
        z_start = results['breaks'][i]
        z_end = results['breaks'][i + 1]

        # Get depths in this segment (new_depths ensures breaks are included)
        segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
        segment_depths = results['new_depths'][segment_mask]
        segment_tt = results['fitted_values_for_plot'][segment_mask]

        # Plot segment
        axs[4].plot(segment_tt, segment_depths, color=colors(i))

    axs[4].set_xlabel('Travel Time (ms)')
    axs[4].set_xlim(0, np.max(TT) * 1.1)
    if np.max(TT) > 150:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
    elif np.max(TT) > 90:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
    elif np.max(TT) > 50:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
    else:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
    axs[4].grid(True, which="both", ls="--")
    axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[5].plot(p['global_velocity'], CPT_depth, label='Zhang et al.', color='tab:blue', linewidth = 1.2)
    axs[5].plot(p['site_specific_velocity'], CPT_depth, label='Zhang et al. (ss)', color='tab:orange', linewidth = 1.2)
    axs[5].plot(p['Robertson_Vs'], p['CPT_depth_rob'], label='Robertson (2012)', color='grey', linewidth = 1.2)
    axs[5].plot(interleave(p['LBR_velocity'], p['LBR_velocity']), interleave(p['LBR_ztop'], p['LBR_zbot']), label='LBR', color='black', linewidth = 1.2)
    axs[5].scatter(SL_Vs, SL_depth, label='Suspension Logging', color='tab:green', s=10)
    axs[5].set_xlim(0, 1000)
    axs[5].set_xticks(np.arange(0, 1001, 200))
    axs[5].set_xlabel('Velocity (m/s)')
    axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
    axs[5].grid(True, which="both", ls="--")

    # Share common y-axis limits across all subplots
    y_max = np.nanmax([np.nanmax(CPT_depth), np.nanmax(p['zbot_TT']), np.nanmax(TT_depth)])
    for ax in axs:
        ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

    # Panel labels (a)-(f) in the top right corner of each subplot
    for ax, panel in zip(axs, 'abcdef'):
        ax.text(0.95, 0.97, f'({panel})', transform=ax.transAxes, ha='right', va='top', fontsize=10,
                bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

    plt.subplots_adjust(bottom=0.28, wspace=0.35)
    plt.tight_layout()
    plt.show()

    return fig


### GUI for all four methods

In [10]:
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Button
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

VsZ_input = widgets.FloatText(value=1.0, description='VsZ Direct Interpretation (LBR) (m/s)', style=style)
VsZ_glob_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. (m/s)', style=style)
VsZ_ss_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. (ss) (m/s)', style=style)
VsZ_rob_input = widgets.FloatText(value=1.0, description='VsZ Robertson (2012) (m/s)', style=style)

button_layout = widgets.Layout(width='600px', height='40px')
EXPORT = Button(description='Export Result', layout = button_layout)
MANUAL_REVISE = Button(description='Manual Revision', layout = button_layout)

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>CPT-Vs Interpretation Tool</b>'),
    HBox([VsZ_input, VsZ_glob_input, VsZ_ss_input, VsZ_rob_input], layout=Layout(width='100%')),
    HBox([EXPORT, MANUAL_REVISE], layout=Layout(width='100%')),
    HTML('<center><font size="+0.5">Slope Break Locations - Direct Interpretation'),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

def Run_and_Plot(slope_break_method = 0, breakpoints = None):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    plt.close('all')
    p = compute_profiles(CPT_depth, CPT_qt, CPT_fs, dGWT, TT_depth, Meas_TT, geology,
                         slope_break_method = slope_break_method, breakpoints = breakpoints)

    print(f"geology = {geology}  |  a_geo = {p['a_geo']:.4f}  |  a_ss = {p['a_site_specific']:.4f}")
    fig = plot_profiles(p)
    fig.savefig('cpt_vs_interpretation.png', dpi=300, bbox_inches='tight')

    VsZ_results = compute_VsZ(p)
    VsZ_input.value = np.round(VsZ_results['LBR'][0], 0)
    VsZ_glob_input.value = np.round(VsZ_results['Zhang et al.'][0], 0)
    VsZ_ss_input.value = np.round(VsZ_results['Zhang et al. (ss)'][0], 0)
    VsZ_rob_input.value = np.round(VsZ_results['Robertson (2012)'][0], 0)

    velocity_updated = p['LBR_velocity']
    top_depth_updated = p['LBR_ztop']
    bottom_depth_updated = p['LBR_zbot']

    return p

# --- Reactive SCPT update ---
def Update_SCPT(_):
    out.clear_output()
    with out:
        p = Run_and_Plot(slope_break_method = 0)

        # Update the checkboxes dynamically
        results = p['results']
        breaks = sorted(set(results.get('candidate_breakpoints', [])))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        checkbox_box.children = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in breaks
        ]

def Full_Manual(_):
    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        try:
            Run_and_Plot(slope_break_method = 1, breakpoints = selected_breakpoints)
        except ValueError as e:
            print(f"Manual revision failed: {e}")

# Export Function
def Export (_):

    # Save the LBR Vs profile to a CSV file
    try:
        VS_SLOPEBREAK_DF = pd.read_csv('slopebreak_vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['Velocity', 'Top_depth', 'Bottom_depth'])

    new_data2 = pd.DataFrame({
        'Velocity': velocity_updated,
        'Top_depth': top_depth_updated,
        'Bottom_depth': bottom_depth_updated})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    VS_SLOPEBREAK_DF.to_csv('slopebreak_vs.csv', index=False)

# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach triggers to the buttons
EXPORT.on_click(Export)
MANUAL_REVISE.on_click(Full_Manual)

Output()